Attemping to use the ann landers file scraper

In [1]:
"""
Dear Abby Archive Scraper

Scrapes advice columns from https://www.uexpress.com/life/dearabby/
"""

import json
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import requests
from bs4 import BeautifulSoup
from dateutil.parser import parse as parse_date


class DearAbbyScraper:
    """Scraper for Dear Abby advice columns."""

    BASE_URL = "https://www.uexpress.com/life/dearabby"

    def __init__(self, data_dir: str = "data/dear_abby", delay: float = 2.5):
        self.data_dir = Path(data_dir)
        self.delay = delay
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
        })
        self.scraped_dates: set[str] = set()
        self._load_scraped_dates()

    def _load_scraped_dates(self):
        """Load already-scraped dates for resume capability."""
        if not self.data_dir.exists():
            return

        for year_dir in self.data_dir.iterdir():
            if year_dir.is_dir() and year_dir.name.isdigit():
                for json_file in year_dir.glob("*.json"):
                    # Extract date from filename (YYYY-MM-DD.json)
                    date_str = json_file.stem
                    self.scraped_dates.add(date_str)

        print(f"Found {len(self.scraped_dates)} already-scraped dates")

    def _get_output_path(self, date_str: str) -> Path:
        """Get output path for a date (YYYY-MM-DD)."""
        year = date_str[:4]
        output_dir = self.data_dir / year
        output_dir.mkdir(parents=True, exist_ok=True)
        return output_dir / f"{date_str}.json"

    def fetch_column(self, date_str: str) -> Optional[dict]:
        """
        Fetch and parse a single Dear Abby column.

        Args:
            date_str: Date in YYYY-MM-DD format

        Returns:
            Parsed column data or None if not found
        """
        # Convert YYYY-MM-DD to URL format YYYY/MM/DD
        url_date = date_str.replace("-", "/")
        url = f"{self.BASE_URL}/{url_date}"

        try:
            response = self.session.get(url, timeout=30)

            if response.status_code == 404:
                return None

            response.raise_for_status()

            return self._parse_column(response.text, date_str, url)

        except requests.exceptions.RequestException as e:
            print(f"  Error fetching {date_str}: {e}")
            return None

    def _parse_column(self, html: str, date_str: str, url: str) -> Optional[dict]:
        """Parse HTML content to extract Q&A pairs."""
        soup = BeautifulSoup(html, "lxml")

        result = {
            "date": date_str,
            "url": url,
            "scraped_at": datetime.now().isoformat(),
            "qa_pairs": [],
            "topics": [],
            "full_text": "",
        }

        # Try to extract JSON-LD metadata first
        self._extract_metadata(soup, result)

        # Find ALL article elements (each contains a separate Q&A pair)
        articles = soup.find_all("article")

        all_text_parts = []

        for article in articles:
            # Extract text from this article
            paragraphs = article.find_all("p")
            article_text = "\n\n".join(
                p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
            )

            if not article_text or "DEAR ABBY" not in article_text.upper():
                continue

            all_text_parts.append(article_text)

            # Parse Q&A pairs from this article's text
            pairs = self._parse_qa_pairs(article_text)
            result["qa_pairs"].extend(pairs)

        result["full_text"] = "\n\n---\n\n".join(all_text_parts)

        if not result["qa_pairs"]:
            # Fallback: try the old method of finding any article content
            article_content = self._find_article_content(soup)
            if article_content:
                paragraphs = article_content.find_all("p")
                full_text = "\n\n".join(
                    p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
                )
                result["full_text"] = full_text
                result["qa_pairs"] = self._parse_qa_pairs(full_text)

        if not result["qa_pairs"]:
            print(f"  Warning: Could not find Q&A pairs for {date_str}")
            return None

        return result

    def _find_article_content(self, soup: BeautifulSoup):
        """Find the main article content container."""
        # Try multiple selectors based on common patterns
        selectors = [
            "article .article-content",
            "article .content",
            ".article-body",
            ".post-content",
            "article",
            ".feature-content",
            'div[class*="article"]',
            'div[class*="content"]',
        ]

        for selector in selectors:
            content = soup.select_one(selector)
            if content and len(content.get_text(strip=True)) > 100:
                return content

        # Fallback: look for div with substantial text containing "DEAR ABBY"
        for div in soup.find_all("div"):
            text = div.get_text()
            if "DEAR ABBY" in text and len(text) > 200:
                return div

        return None

    def _extract_metadata(self, soup: BeautifulSoup, result: dict):
        """Extract metadata from JSON-LD or other sources."""
        # Look for JSON-LD script tags
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, dict):
                    # Extract topics/keywords
                    if "keywords" in data:
                        keywords = data["keywords"]
                        if isinstance(keywords, str):
                            result["topics"] = [k.strip() for k in keywords.split(",")]
                        elif isinstance(keywords, list):
                            result["topics"] = keywords
            except (json.JSONDecodeError, TypeError):
                pass

        # Also check for meta tags
        keywords_meta = soup.find("meta", {"name": "keywords"})
        if keywords_meta and keywords_meta.get("content") and not result["topics"]:
            result["topics"] = [k.strip() for k in keywords_meta["content"].split(",")]

    def _parse_qa_pairs(self, text: str) -> list[dict]:
        """Parse Q&A pairs from column text."""
        pairs = []

        # Pattern to match "DEAR ABBY:" followed by content
        # Then "DEAR [NAME]:" for the response

        # Split on "DEAR ABBY:" to get individual letters
        # But preserve the delimiter
        parts = re.split(r'(DEAR ABBY:)', text, flags=re.IGNORECASE)

        i = 0
        while i < len(parts):
            # Look for "DEAR ABBY:" marker
            if parts[i].strip().upper() == "DEAR ABBY:":
                if i + 1 < len(parts):
                    content = parts[i + 1]

                #response_match = re.search(
                    #r'(celibi\s+[A-Z"][A-Z\s\-\'\.!\d"]+(?:\s+IN\s+[A-Z\.]+)?:)',
                    #content,
                    #flags=re.IGNORECASE
                #)

                    # Find the response - starts with "DEAR " followed by a name
                    # Common patterns: "DEAR WORRIED:", "DEAR ANONYMOUS:", "DEAR HELP!:", "DEAR \"90\":", etc.
                    # Allow letters, spaces, hyphens, apostrophes, periods, exclamation marks, quotes, digits
                    response_match = re.search(
                        r'(DEAR\s+[A-Z"][A-Z\s\-\'\.!\d"]+(?:\s+IN\s+[A-Z\.]+)?:)',
                        content,
                        flags=re.IGNORECASE
                    )
                    

                    if response_match:
                        letter = content[:response_match.start()].strip()
                        response = content[response_match.start():].strip()

                        # Extract the signature/name from the response header
                        response_header = response_match.group(1)
                        signature = response_header.replace("DEAR", "").replace(":", "").strip()

                        # Check if there's another "DEAR ABBY:" in the response
                        # which would indicate the response ends there
                        next_letter = re.search(r'DEAR ABBY:', response, flags=re.IGNORECASE)
                        if next_letter:
                            response = response[:next_letter.start()].strip()

                        pairs.append({
                            "letter": f"DEAR ABBY: {letter}",
                            "response": response,
                            "letter_signature": signature,
                        })
                    else:
                        # No clear response pattern found, store as unparsed
                        pairs.append({
                            "letter": f"DEAR ABBY: {content.strip()}",
                            "response": "",
                            "letter_signature": "",
                        })
                i += 2
            else:
                i += 1

        return pairs

    def scrape_date_range(self, start_date: str, end_date: str):
        """
        Scrape all columns in a date range.

        Args:
            start_date: Start date in YYYY-MM-DD format
            end_date: End date in YYYY-MM-DD format
        """
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(end_date, "%Y-%m-%d")

        current = start
        total_scraped = 0
        total_skipped = 0
        total_not_found = 0

        print(f"Scraping from {start_date} to {end_date}")
        print(f"Delay between requests: {self.delay}s")
        print("-" * 50)

        while current <= end:
            date_str = current.strftime("%Y-%m-%d")

            # Check if already scraped (resume capability)
            if date_str in self.scraped_dates:
                total_skipped += 1
                current += timedelta(days=1)
                continue

            print(f"Fetching {date_str}...", end=" ")

            result = self.fetch_column(date_str)

            if result:
                # Save the result
                output_path = self._get_output_path(date_str)
                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)

                qa_count = len(result["qa_pairs"])
                print(f"OK ({qa_count} Q&A pairs)")
                self.scraped_dates.add(date_str)
                total_scraped += 1
            else:
                print("Not found")
                total_not_found += 1

            # Rate limiting
            time.sleep(self.delay)
            current += timedelta(days=1)

        print("-" * 50)
        print(f"Done! Scraped: {total_scraped}, Skipped: {total_skipped}, Not found: {total_not_found}")


def main():
    import argparse

    parser = argparse.ArgumentParser(description="Scrape Dear Abby columns")
    parser.add_argument("--start-date", required=True, help="Start date (YYYY-MM-DD)")
    parser.add_argument("--end-date", required=True, help="End date (YYYY-MM-DD)")
    parser.add_argument("--delay", type=float, default=2.5, help="Delay between requests (seconds)")
    parser.add_argument("--data-dir", default="data/dear_abby", help="Output directory")

    args = parser.parse_args()

    scraper = DearAbbyScraper(data_dir=args.data_dir, delay=args.delay)
    scraper.scrape_date_range(args.start_date, args.end_date)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --start-date START_DATE --end-date END_DATE
                             [--delay DELAY] [--data-dir DATA_DIR]
ipykernel_launcher.py: error: the following arguments are required: --start-date, --end-date


SystemExit: 2

/Users/aluisek/Desktop/Github/Ace-Research/projectPython/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
